# Верификация LLM-разметки

Ручная проверка выборки пар из `labeled_pairs.parquet`.

**Стратегии выборки:**
1. **Случайные match (high conf)** — baseline точности положительного класса
2. **No-match с высоким similarity (high conf)** — ложные отклонения?
3. **Все medium-confidence** — пограничные случаи
4. **Match с низким similarity** — подозрительные match

**Workflow:** нажимай кнопки Match / No Match / Skip, результат сохраняется автоматически.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_DIR = Path("../data/labeled")
LABELED_PATH = DATA_DIR / "labeled_pairs.parquet"
VERIFY_PATH = DATA_DIR / "verification.parquet"

assert LABELED_PATH.exists(), f"Файл {LABELED_PATH} не найден. Сначала запустите 12_label_real_data.py"

df = pd.read_parquet(LABELED_PATH)
done = df[df["label"] != -1].copy()
print(f"Загружено: {len(done):,} размеченных пар")
print(f"  Match: {(done['label']==1).sum():,}  No match: {(done['label']==0).sum():,}")
print(f"  Confidence: {done['confidence'].value_counts().to_dict()}")

Загружено: 26,055 размеченных пар
  Match: 2,541  No match: 23,514
  Confidence: {'high': 25366, 'medium': 689}


## 1. Формирование выборки для проверки

In [2]:
rng = np.random.RandomState(42)

samples = {}

# 1. Случайные match (high confidence)
match_high = done[(done["label"] == 1) & (done["confidence"] == "high")]
samples["match_high_random"] = match_high.sample(n=min(50, len(match_high)), random_state=rng)

# 2. No-match с высоким similarity (high confidence) — потенциальные ошибки
nomatch_high_sim = done[(done["label"] == 0) & (done["confidence"] == "high")].nlargest(50, "cosine_sim")
samples["nomatch_high_sim"] = nomatch_high_sim

# 3. Все medium-confidence пары
medium = done[done["confidence"] == "medium"]
samples["medium_conf"] = medium if len(medium) <= 200 else medium.sample(n=200, random_state=rng)

# 4. Match с низким similarity — подозрительные
match_low_sim = done[done["label"] == 1].nsmallest(50, "cosine_sim")
samples["match_low_sim"] = match_low_sim

# 5. Случайные no-match (контрольная группа)
nomatch_random = done[(done["label"] == 0) & (done["confidence"] == "high")]
samples["nomatch_random"] = nomatch_random.sample(n=min(50, len(nomatch_random)), random_state=rng)

# Объединяем, убираем дубли
verify_df = pd.concat(samples.values()).drop_duplicates(subset=["spec_text", "nom_text"])
verify_df = verify_df.copy()
verify_df["strategy"] = ""
for name, sub in samples.items():
    mask = verify_df.index.isin(sub.index)
    verify_df.loc[mask & (verify_df["strategy"] == ""), "strategy"] = name

verify_df = verify_df.reset_index(drop=True)
verify_df["human_label"] = -1  # -1 = не проверено, 1 = match, 0 = no match, 2 = skip/unsure

# Загрузить предыдущий прогресс если есть
if VERIFY_PATH.exists():
    prev = pd.read_parquet(VERIFY_PATH)
    prev_map = prev.set_index(["spec_text", "nom_text"])["human_label"]
    for i, row in verify_df.iterrows():
        key = (row["spec_text"], row["nom_text"])
        if key in prev_map.index:
            verify_df.at[i, "human_label"] = prev_map[key]
    already = (verify_df["human_label"] != -1).sum()
    print(f"Загружен предыдущий прогресс: {already} из {len(verify_df)} уже проверено")

print(f"\nВсего пар для проверки: {len(verify_df)}")
print(f"Осталось проверить: {(verify_df['human_label'] == -1).sum()}")
print(f"\nПо стратегиям:")
for name in samples:
    n = (verify_df["strategy"] == name).sum()
    print(f"  {name:25s}: {n}")


Всего пар для проверки: 398
Осталось проверить: 398

По стратегиям:
  match_high_random        : 50
  nomatch_high_sim         : 50
  medium_conf              : 200
  match_low_sim            : 48
  nomatch_random           : 50


## 2. Виджет для ручной разметки

Нажимай кнопки:
- **Match** — пара описывает один и тот же товар
- **No Match** — разные товары
- **Skip** — непонятно / недостаточно информации
- **Back** — вернуться к предыдущей паре

Прогресс сохраняется автоматически каждые 10 пар и при нажатии **Save**.

In [3]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

class VerificationWidget:
    AUTOSAVE_EVERY = 10

    def __init__(self, data: pd.DataFrame, save_path: Path):
        self.data = data
        self.save_path = save_path
        # Start from first unlabeled
        unlabeled = data.index[data["human_label"] == -1].tolist()
        self.idx = unlabeled[0] if unlabeled else 0
        self.actions_since_save = 0

        # --- UI elements ---
        self.pair_html = widgets.HTML()
        self.progress_label = widgets.HTML()
        self.status_label = widgets.HTML()

        btn_match = widgets.Button(description="Match", button_style="success",
                                   icon="check", layout=widgets.Layout(width="120px", height="40px"))
        btn_nomatch = widgets.Button(description="No Match", button_style="danger",
                                     icon="times", layout=widgets.Layout(width="120px", height="40px"))
        btn_skip = widgets.Button(description="Skip", button_style="warning",
                                  icon="forward", layout=widgets.Layout(width="120px", height="40px"))
        btn_back = widgets.Button(description="Back", button_style="info",
                                  icon="arrow-left", layout=widgets.Layout(width="120px", height="40px"))
        btn_save = widgets.Button(description="Save", button_style="",
                                  icon="save", layout=widgets.Layout(width="120px", height="40px"))

        btn_match.on_click(lambda _: self._label(1))
        btn_nomatch.on_click(lambda _: self._label(0))
        btn_skip.on_click(lambda _: self._label(2))
        btn_back.on_click(lambda _: self._go_back())
        btn_save.on_click(lambda _: self._save(force=True))

        self.buttons = widgets.HBox([btn_match, btn_nomatch, btn_skip, btn_back, btn_save],
                                    layout=widgets.Layout(justify_content="center", gap="8px"))

        # Navigation slider
        self.slider = widgets.IntSlider(value=self.idx, min=0, max=len(data) - 1,
                                        description="Пара #:", continuous_update=False,
                                        layout=widgets.Layout(width="80%"))
        self.slider.observe(self._on_slider, names="value")

        # Filter dropdown
        self.filter_dd = widgets.Dropdown(
            options=[("Все", "all"), ("Не проверенные", "unlabeled"),
                     ("match_high_random", "match_high_random"),
                     ("nomatch_high_sim", "nomatch_high_sim"),
                     ("medium_conf", "medium_conf"),
                     ("match_low_sim", "match_low_sim"),
                     ("nomatch_random", "nomatch_random")],
            value="all", description="Фильтр:", layout=widgets.Layout(width="350px"))
        self.filter_dd.observe(self._on_filter, names="value")

        self.container = widgets.VBox([
            self.progress_label,
            widgets.HBox([self.filter_dd], layout=widgets.Layout(justify_content="center")),
            self.slider,
            self.pair_html,
            self.buttons,
            self.status_label,
        ])

        self._update_display()

    def _render_pair(self, row: pd.Series) -> str:
        llm_label = "MATCH" if row["label"] == 1 else "NO MATCH"
        llm_color = "#27ae60" if row["label"] == 1 else "#e74c3c"
        human = row["human_label"]
        human_str = {-1: "---", 0: "NO MATCH", 1: "MATCH", 2: "SKIP"}.get(human, "?")
        human_color = {-1: "#888", 0: "#e74c3c", 1: "#27ae60", 2: "#f39c12"}.get(human, "#888")

        agree = ""
        if human in (0, 1):
            if human == row["label"]:
                agree = ' <span style="color:#27ae60;font-weight:bold">AGREE</span>'
            else:
                agree = ' <span style="color:#e74c3c;font-weight:bold">DISAGREE</span>'

        return f"""
        <div style="border:2px solid #ddd; border-radius:12px; padding:20px; max-width:900px;
                    margin:10px auto; background:#fafafa; font-family:monospace;">
          <div style="display:flex; justify-content:space-between; margin-bottom:12px;">
            <span style="font-size:13px; color:#888;">Стратегия: <b>{row.get('strategy','?')}</b></span>
            <span style="font-size:13px;">cosine sim = <b>{row['cosine_sim']:.4f}</b></span>
          </div>
          <table style="width:100%; border-collapse:collapse;">
            <tr style="background:#e8f4fd;">
              <td style="padding:10px; width:100px; font-weight:bold; color:#2980b9;">Спецификация</td>
              <td style="padding:10px; font-size:14px; word-break:break-all;">{row['spec_text']}</td>
            </tr>
            <tr style="background:#fdf2e9;">
              <td style="padding:10px; font-weight:bold; color:#e67e22;">Номенклатура</td>
              <td style="padding:10px; font-size:14px; word-break:break-all;">{row['nom_text']}</td>
            </tr>
          </table>
          <div style="margin-top:12px; display:flex; justify-content:space-around;">
            <span>LLM: <b style="color:{llm_color};">{llm_label}</b>
                  (conf: {row.get('confidence','?')})</span>
            <span>Human: <b style="color:{human_color};">{human_str}</b>{agree}</span>
          </div>
        </div>"""

    def _update_display(self):
        row = self.data.iloc[self.idx]
        self.pair_html.value = self._render_pair(row)

        total = len(self.data)
        checked = (self.data["human_label"] != -1).sum()
        agree = ((self.data["human_label"] != -1) & (self.data["human_label"] != 2) &
                 (self.data["human_label"] == self.data["label"])).sum()
        disagree = ((self.data["human_label"] != -1) & (self.data["human_label"] != 2) &
                    (self.data["human_label"] != self.data["label"])).sum()
        skipped = (self.data["human_label"] == 2).sum()

        pct = 100 * checked / total if total > 0 else 0
        self.progress_label.value = (
            f'<div style="text-align:center; font-size:14px; margin:5px;">'
            f'Прогресс: <b>{checked}/{total}</b> ({pct:.0f}%) &nbsp;|&nbsp; '
            f'Agree: <b style="color:#27ae60">{agree}</b> &nbsp; '
            f'Disagree: <b style="color:#e74c3c">{disagree}</b> &nbsp; '
            f'Skip: <b style="color:#f39c12">{skipped}</b> &nbsp;|&nbsp; '
            f'Текущая: <b>#{self.idx}</b>'
            f'</div>')

        self.slider.value = self.idx

    def _label(self, value: int):
        self.data.at[self.idx, "human_label"] = value
        self.actions_since_save += 1

        if self.actions_since_save >= self.AUTOSAVE_EVERY:
            self._save()

        # Advance to next unlabeled
        remaining = self.data.index[(self.data.index > self.idx) & (self.data["human_label"] == -1)]
        if len(remaining) > 0:
            self.idx = remaining[0]
        elif self.idx < len(self.data) - 1:
            self.idx += 1

        self._update_display()

    def _go_back(self):
        if self.idx > 0:
            self.idx -= 1
            self._update_display()

    def _on_slider(self, change):
        self.idx = change["new"]
        self._update_display()

    def _on_filter(self, change):
        filt = change["new"]
        if filt == "all":
            subset = self.data.index
        elif filt == "unlabeled":
            subset = self.data.index[self.data["human_label"] == -1]
        else:
            subset = self.data.index[self.data["strategy"] == filt]

        if len(subset) > 0:
            self.idx = subset[0]
            self._update_display()
        else:
            self.status_label.value = '<div style="text-align:center;color:#e74c3c;">Нет пар в этой категории</div>'

    def _save(self, force=False):
        self.data.to_parquet(self.save_path)
        self.actions_since_save = 0
        self.status_label.value = (
            f'<div style="text-align:center; color:#27ae60; font-size:12px;">'
            f'Сохранено в {self.save_path.name}</div>')

    def show(self):
        display(self.container)


w = VerificationWidget(verify_df, VERIFY_PATH)
w.show()

## 3. Статистика верификации

Запусти эту ячейку после завершения разметки (или в процессе — для промежуточных результатов).

In [4]:
# Загрузить актуальные данные (из виджета или с диска)
if VERIFY_PATH.exists():
    vdf = pd.read_parquet(VERIFY_PATH)
else:
    vdf = verify_df.copy()

checked = vdf[vdf["human_label"].isin([0, 1])].copy()  # без skip
print(f"Проверено (без skip): {len(checked)} / {len(vdf)}")
print(f"Skip: {(vdf['human_label'] == 2).sum()}")
print(f"Осталось: {(vdf['human_label'] == -1).sum()}")

if len(checked) > 0:
    checked["agree"] = checked["human_label"] == checked["label"]

    print(f"\n{'='*50}")
    print(f"ОБЩАЯ ТОЧНОСТЬ LLM: {checked['agree'].mean():.1%} ({checked['agree'].sum()}/{len(checked)})")
    print(f"{'='*50}")

    # По стратегиям
    print(f"\nПо стратегиям:")
    for strat in checked["strategy"].unique():
        sub = checked[checked["strategy"] == strat]
        acc = sub["agree"].mean()
        n = len(sub)
        n_disagree = (~sub["agree"]).sum()
        print(f"  {strat:25s}: accuracy={acc:.1%}  ({n_disagree} ошибок из {n})")

    # По типу ошибки
    false_match = checked[(checked["label"] == 1) & (checked["human_label"] == 0)]
    false_nomatch = checked[(checked["label"] == 0) & (checked["human_label"] == 1)]
    print(f"\nТипы ошибок LLM:")
    print(f"  False match (LLM=match, human=no):    {len(false_match)}")
    print(f"  False no-match (LLM=no, human=match): {len(false_nomatch)}")

    # По confidence
    print(f"\nПо confidence:")
    for conf in ["high", "medium", "low"]:
        sub = checked[checked["confidence"] == conf]
        if len(sub) > 0:
            acc = sub["agree"].mean()
            print(f"  {conf:8s}: accuracy={acc:.1%}  ({len(sub)} пар)")

    if len(false_match) > 0:
        print(f"\n--- Примеры False Match (LLM ошибочно поставил match) ---")
        for _, r in false_match.head(5).iterrows():
            print(f"  sim={r['cosine_sim']:.3f} SPEC: {r['spec_text'][:65]}")
            print(f"{'':>17s} NOM:  {r['nom_text'][:65]}\n")

    if len(false_nomatch) > 0:
        print(f"\n--- Примеры False No-Match (LLM ошибочно отклонил) ---")
        for _, r in false_nomatch.head(5).iterrows():
            print(f"  sim={r['cosine_sim']:.3f} SPEC: {r['spec_text'][:65]}")
            print(f"{'':>17s} NOM:  {r['nom_text'][:65]}\n")
else:
    print("Ещё ничего не проверено — размечай пары в виджете выше")

Проверено (без skip): 398 / 398
Skip: 0
Осталось: 0

ОБЩАЯ ТОЧНОСТЬ LLM: 70.9% (282/398)

По стратегиям:
  match_high_random        : accuracy=86.0%  (7 ошибок из 50)
  nomatch_high_sim         : accuracy=60.0%  (20 ошибок из 50)
  medium_conf              : accuracy=61.5%  (77 ошибок из 200)
  match_low_sim            : accuracy=77.1%  (11 ошибок из 48)
  nomatch_random           : accuracy=98.0%  (1 ошибок из 50)

Типы ошибок LLM:
  False match (LLM=match, human=no):    61
  False no-match (LLM=no, human=match): 55

По confidence:
  high    : accuracy=79.3%  (188 пар)
  medium  : accuracy=63.3%  (210 пар)

--- Примеры False Match (LLM ошибочно поставил match) ---
  sim=0.936 SPEC: 0,125-9,1 кОм ± 5 % -А-Д-В | ОЖ0.467.093 ТУ
                  NOM:  Р1-12-0,125-9,1 К  5% | Артикул: 412050

  sim=0.905 SPEC: 100 пФ+-5 % 100 В | Код: 27.90.60.000 | ТУРБ 14576608.003-96
                  NOM:  К10-17А-МПО-100В-100 5% т1 | Артикул: 420837

  sim=0.945 SPEC: Н20-0,033 мкФ± 10% | ОЖ0.460.174